# Silver Player Identity, Participation, and Grain Validation

## tl;dr

For the 2023–2025 WR slice, GSIS is the canonical player identifier: the player master contains 24,828 unique, non-null GSIS IDs, and every non-null weekly-stat player ID and PBP receiver ID matches it. Weekly roster snapshots contain a small set of valid-looking GSIS IDs absent from the player master, so those records should be retained as identity exceptions rather than discarded.

Snap counts are the best available participation evidence. Among mapped WR snap rows, nearly every row records at least one offensive, defensive, or special-teams snap, while weekly stats omit many players who took snaps but recorded no box-score event. Roster status remains useful context but does not by itself prove game participation.

The recommended weekly row universe is the union of identified weekly-stat and snap-count observations. Weekly rosters enrich those rows with status and position context but do not create facts for reserve, development-squad, or other non-participating players. The proposed grain is `player_id + season + week + team`; `game_id`, opponent, detailed game type, and normalized `REG`/`POST` season type remain attributes. This notebook validates and documents these rules but writes no Silver Parquet files.

## Context & Methods

### Scope

- Seasons: 2023–2025
- First vertical slice: wide receivers
- Notebook mode: data-quality and contract validation
- Inputs: existing local Bronze Parquet files only
- Output: evidence, exceptions, and decisions; no persisted Silver dataset

This notebook answers the identity, participation, and grain questions that must be stable before shared Silver facts are assembled. It uses weekly evidence rather than assuming the player master's latest position describes every historical week.

### Key Assumptions

- GSIS identifiers are accepted only through exact identifier matches; player names are never join keys.
- A snap-count record must resolve through a one-to-one PFR-to-GSIS bridge before it can enter a GSIS-grained dataset.
- A player participated in a game only when the source records at least one offensive, defensive, or special-teams snap.
- Missing participation evidence is unknown, not an observed zero.
- `WC`, `DIV`, `CON`, and `SB` retain their detail but normalize to `POST` for compatibility with weekly stats.
- The shared Silver design should remain usable beyond WR, even though this notebook validates the contracts with the WR slice.

## Data

### 1. Setup

In [1]:
from pathlib import Path

import pandas as pd

SEASONS = [2023, 2024, 2025]
POSTSEASON_GAME_TYPES = {"WC", "DIV", "CON", "SB"}
WEEKLY_GRAIN = ["player_id", "season", "week", "team"]

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 180)

# Find the repository root whether this runs from Jupyter or nbconvert.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "ROADMAP.md").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / "ROADMAP.md").exists():
            PROJECT_ROOT = parent
            break
    else:
        raise RuntimeError("Could not locate the repository root.")

BRONZE_DIR = PROJECT_ROOT / "data/bronze"
PROJECT_ROOT

PosixPath('/Users/dtwice/Development/dev_sports/nfl/fantasy-football/nfl-dream-lab-app')

### 2. Load the persisted Bronze sources

Play-by-play is restricted to the columns needed for receiver identity coverage. The complete 372-column files are not needed for this validation.

In [2]:
players_path = BRONZE_DIR / "players/players.parquet"
schedule_paths = sorted((BRONZE_DIR / "schedules").glob("*.parquet"))
player_stats_paths = sorted((BRONZE_DIR / "player_stats_weekly").glob("*.parquet"))
roster_paths = sorted((BRONZE_DIR / "rosters_weekly").glob("*.parquet"))
snap_paths = sorted((BRONZE_DIR / "snap_counts").glob("*.parquet"))
pbp_paths = sorted((BRONZE_DIR / "pbp").glob("*.parquet"))

assert players_path.exists()
assert len(schedule_paths) == len(SEASONS)
assert len(player_stats_paths) == len(SEASONS)
assert len(roster_paths) == len(SEASONS)
assert len(snap_paths) == len(SEASONS)
assert len(pbp_paths) == len(SEASONS)

players = pd.read_parquet(players_path)
schedules = pd.concat(
    [pd.read_parquet(path) for path in schedule_paths],
    ignore_index=True,
)
player_stats = pd.concat(
    [pd.read_parquet(path) for path in player_stats_paths],
    ignore_index=True,
)
rosters = pd.concat(
    [pd.read_parquet(path) for path in roster_paths],
    ignore_index=True,
)
snap_counts = pd.concat(
    [pd.read_parquet(path) for path in snap_paths],
    ignore_index=True,
)
pbp_identity_columns = [
    "season",
    "season_type",
    "week",
    "game_id",
    "posteam",
    "receiver_player_id",
]
pbp_receivers = pd.concat(
    [pd.read_parquet(path, columns=pbp_identity_columns) for path in pbp_paths],
    ignore_index=True,
)

In [3]:
input_summary = pd.DataFrame(
    [
        ["players", len(players), len(players.columns), 1],
        ["schedules", len(schedules), len(schedules.columns), len(schedule_paths)],
        ["player_stats_weekly", len(player_stats), len(player_stats.columns), len(player_stats_paths)],
        ["rosters_weekly", len(rosters), len(rosters.columns), len(roster_paths)],
        ["snap_counts", len(snap_counts), len(snap_counts.columns), len(snap_paths)],
        ["pbp identity columns", len(pbp_receivers), len(pbp_receivers.columns), len(pbp_paths)],
    ],
    columns=["dataset", "rows", "loaded_columns", "files"],
)

assert set(schedules["season"].dropna().unique()) == set(SEASONS)
assert set(player_stats["season"].dropna().unique()) == set(SEASONS)
assert set(rosters["season"].dropna().unique()) == set(SEASONS)
assert set(snap_counts["season"].dropna().unique()) == set(SEASONS)
assert set(pbp_receivers["season"].dropna().unique()) == set(SEASONS)

input_summary

,dataset,rows,loaded_columns,files
0,players,24828,39,1
1,schedules,855,46,3
2,player_stats_weekly,57048,150,3
3,rosters_weekly,139083,36,3
4,snap_counts,79767,16,3
5,pbp identity columns,147928,6,3


## Results

### 3. Normalize game and team context

Schedules retain playoff-round detail. A second `season_type` field provides the common `REG`/`POST` vocabulary used by weekly stats. Each scheduled team should appear at most once per season and week.

In [4]:
home_team_games = schedules[
    ["game_id", "season", "week", "game_type", "home_team", "away_team"]
].rename(columns={"home_team": "team", "away_team": "opponent_team"})

away_team_games = schedules[
    ["game_id", "season", "week", "game_type", "away_team", "home_team"]
].rename(columns={"away_team": "team", "home_team": "opponent_team"})

schedule_team_games = pd.concat(
    [home_team_games, away_team_games],
    ignore_index=True,
)
schedule_team_games["season_type"] = schedule_team_games["game_type"].map(
    lambda game_type: "POST" if game_type in POSTSEASON_GAME_TYPES else game_type
)

team_week_duplicate_rows = schedule_team_games.duplicated(
    ["team", "season", "week"]
).sum()

assert set(schedule_team_games["season_type"].unique()) == {"REG", "POST"}
assert team_week_duplicate_rows == 0

schedule_context_summary = (
    schedule_team_games.groupby(["season", "game_type", "season_type"], dropna=False)
    .agg(
        team_games=("game_id", "size"),
        teams=("team", "nunique"),
        min_week=("week", "min"),
        max_week=("week", "max"),
    )
    .reset_index()
)

schedule_context_summary

,season,game_type,season_type,team_games,teams,min_week,max_week
0,2023,CON,POST,4,4,21,21
1,2023,DIV,POST,8,8,20,20
2,2023,REG,REG,544,32,1,18
3,2023,SB,POST,2,2,22,22
4,2023,WC,POST,12,12,19,19
5,2024,CON,POST,4,4,21,21
6,2024,DIV,POST,8,8,20,20
7,2024,REG,REG,544,32,1,18
8,2024,SB,POST,2,2,22,22
9,2024,WC,POST,12,12,19,19


### 4. Validate GSIS as the canonical player identifier

The player master is the preferred descriptive dimension, but source rows with a populated GSIS ID are not discarded solely because that ID is absent from the current master snapshot. Such rows remain visible as identity exceptions.

In [5]:
player_stats_with_id = player_stats.loc[player_stats["player_id"].notna()].copy()
rosters_with_id = rosters.loc[rosters["gsis_id"].notna()].copy()
pbp_with_receiver = pbp_receivers.loc[pbp_receivers["receiver_player_id"].notna()].copy()

identity_summary = pd.DataFrame(
    [
        {
            "source": "players",
            "identifier": "gsis_id",
            "rows_with_id": players["gsis_id"].notna().sum(),
            "distinct_ids": players["gsis_id"].nunique(),
            "null_id_rows": players["gsis_id"].isna().sum(),
            "duplicate_id_rows": players.loc[players["gsis_id"].notna(), "gsis_id"].duplicated().sum(),
            "master_match_rate": 1.0,
        },
        {
            "source": "player_stats_weekly",
            "identifier": "player_id",
            "rows_with_id": len(player_stats_with_id),
            "distinct_ids": player_stats_with_id["player_id"].nunique(),
            "null_id_rows": player_stats["player_id"].isna().sum(),
            "duplicate_id_rows": pd.NA,
            "master_match_rate": player_stats_with_id["player_id"].isin(players["gsis_id"]).mean(),
        },
        {
            "source": "rosters_weekly",
            "identifier": "gsis_id",
            "rows_with_id": len(rosters_with_id),
            "distinct_ids": rosters_with_id["gsis_id"].nunique(),
            "null_id_rows": rosters["gsis_id"].isna().sum(),
            "duplicate_id_rows": pd.NA,
            "master_match_rate": rosters_with_id["gsis_id"].isin(players["gsis_id"]).mean(),
        },
        {
            "source": "pbp receivers",
            "identifier": "receiver_player_id",
            "rows_with_id": len(pbp_with_receiver),
            "distinct_ids": pbp_with_receiver["receiver_player_id"].nunique(),
            "null_id_rows": pbp_receivers["receiver_player_id"].isna().sum(),
            "duplicate_id_rows": pd.NA,
            "master_match_rate": pbp_with_receiver["receiver_player_id"].isin(players["gsis_id"]).mean(),
        },
    ]
)
identity_summary["master_match_percent"] = identity_summary["master_match_rate"].map(
    lambda value: f"{value:.2%}"
)

assert players["gsis_id"].notna().all()
assert not players["gsis_id"].duplicated().any()
assert identity_summary.loc[identity_summary["source"] == "player_stats_weekly", "master_match_rate"].iloc[0] == 1.0
assert identity_summary.loc[identity_summary["source"] == "pbp receivers", "master_match_rate"].iloc[0] == 1.0

identity_summary.drop(columns="master_match_rate")

,source,identifier,rows_with_id,distinct_ids,null_id_rows,duplicate_id_rows,master_match_percent
0,players,gsis_id,24828,24828,0,0,100.00%
1,player_stats_weekly,player_id,56982,2836,66,<NA>,100.00%
2,rosters_weekly,gsis_id,139053,4566,30,<NA>,99.59%
3,pbp receivers,receiver_player_id,53880,759,94048,<NA>,100.00%


In [6]:
roster_identity_exceptions = rosters_with_id.loc[
    ~rosters_with_id["gsis_id"].isin(players["gsis_id"]),
    ["gsis_id", "full_name", "position", "season", "week", "team", "status"],
].copy()

roster_identity_exception_summary = (
    roster_identity_exceptions.groupby("position", dropna=False)
    .agg(
        rows=("gsis_id", "size"),
        distinct_ids=("gsis_id", "nunique"),
    )
    .sort_values("rows", ascending=False)
    .reset_index()
)

print(
    f"Roster rows with a populated GSIS ID missing from the player master: "
    f"{len(roster_identity_exceptions):,} rows / "
    f"{roster_identity_exceptions['gsis_id'].nunique():,} IDs"
)
display(roster_identity_exception_summary.head(12))
display(roster_identity_exceptions.head(15))

Roster rows with a populated GSIS ID missing from the player master: 574 rows / 496 IDs


,position,rows,distinct_ids
0,DB,105,102
1,DL,104,87
2,WR,104,96
3,OL,94,71
4,LB,74,55
5,RB,36,35
6,TE,31,28
7,QB,11,11
8,LS,7,5
9,K,6,4


,gsis_id,full_name,position,season,week,team,status
17019,00-0035435,Rodney Randle,DB,2023,1,NE,CUT
19532,00-0035977,Nevelle Clarke,DB,2023,1,PIT,CUT
19989,00-0036097,Kyahva Tezino,LB,2023,1,SF,CUT
24417,00-0036548,Stevie Scott,RB,2023,1,ARI,CUT
29926,00-0037126,Cole Schneider,OL,2023,1,GB,CUT
32494,00-0037350,Elijah Hamilton,DB,2023,1,GB,CUT
32495,00-0037352,Deandre Johnson,LB,2023,20,GB,RES
32843,00-0037397,Alex Mollette,OL,2023,1,DET,CUT
32976,00-0037427,Carson Wells,LB,2023,1,NE,CUT
33577,00-0037519,Kevin Atkins,DL,2023,1,NYG,CUT


### 5. Validate the PFR-to-GSIS bridge for snap counts

The player master supplies a one-to-one bridge wherever `pfr_id` is present. Roster crosswalk fields are inspected separately because historical roster mappings can be ambiguous and should not silently override the master.

In [7]:
master_pfr_bridge = players.loc[
    players["pfr_id"].notna(),
    ["pfr_id", "gsis_id", "display_name", "position"],
].copy()

assert not master_pfr_bridge["pfr_id"].duplicated().any()
assert not master_pfr_bridge["gsis_id"].duplicated().any()

mapped_snaps = snap_counts.merge(
    master_pfr_bridge,
    left_on="pfr_player_id",
    right_on="pfr_id",
    how="left",
    validate="many_to_one",
    indicator="identity_join",
)

wr_mapped_snaps = mapped_snaps.loc[mapped_snaps["position_x"] == "WR"].copy()

pfr_bridge_summary = pd.DataFrame(
    [
        {
            "population": "all snap rows",
            "rows": len(mapped_snaps),
            "matched_rows": mapped_snaps["identity_join"].eq("both").sum(),
            "unmatched_rows": mapped_snaps["identity_join"].ne("both").sum(),
            "unmatched_pfr_ids": mapped_snaps.loc[mapped_snaps["identity_join"].ne("both"), "pfr_player_id"].nunique(),
        },
        {
            "population": "snap rows labeled WR",
            "rows": len(wr_mapped_snaps),
            "matched_rows": wr_mapped_snaps["identity_join"].eq("both").sum(),
            "unmatched_rows": wr_mapped_snaps["identity_join"].ne("both").sum(),
            "unmatched_pfr_ids": wr_mapped_snaps.loc[wr_mapped_snaps["identity_join"].ne("both"), "pfr_player_id"].nunique(),
        },
    ]
)
pfr_bridge_summary["match_percent"] = (
    pfr_bridge_summary["matched_rows"] / pfr_bridge_summary["rows"]
).map(lambda value: f"{value:.2%}")

pfr_bridge_summary

,population,rows,matched_rows,unmatched_rows,unmatched_pfr_ids,match_percent
0,all snap rows,79767,79656,111,11,99.86%
1,snap rows labeled WR,8862,8848,14,1,99.84%


In [8]:
wr_snap_identity_exceptions = wr_mapped_snaps.loc[
    wr_mapped_snaps["identity_join"].ne("both"),
    ["season", "week", "game_id", "team", "player", "pfr_player_id", "offense_snaps", "st_snaps"],
].sort_values(["season", "week"])

roster_pfr_bridge = rosters.loc[
    rosters["gsis_id"].notna() & rosters["pfr_id"].notna(),
    ["pfr_id", "gsis_id", "full_name"],
].drop_duplicates()
ambiguous_roster_pfr_ids = (
    roster_pfr_bridge.groupby("pfr_id")["gsis_id"].nunique().loc[lambda values: values > 1].index
)
ambiguous_roster_crosswalk = roster_pfr_bridge.loc[
    roster_pfr_bridge["pfr_id"].isin(ambiguous_roster_pfr_ids)
].sort_values(["pfr_id", "gsis_id"])

print(f"Player-master rows without PFR ID: {players['pfr_id'].isna().sum():,}")
print(f"Ambiguous roster PFR IDs: {len(ambiguous_roster_pfr_ids):,}")
display(wr_snap_identity_exceptions.drop_duplicates(["player", "pfr_player_id"]))
display(ambiguous_roster_crosswalk)

Player-master rows without PFR ID: 2,175
Ambiguous roster PFR IDs: 1


,season,week,game_id,team,player,pfr_player_id,offense_snaps,st_snaps
38796,2024,9,2024_09_LA_SEA,SEA,Cody White,WhitCo05,35.0,11.0


,pfr_id,gsis_id,full_name
43652,YounBy01,00-0038978,Byron Young
45082,YounBy01,00-0039137,Byron Young


### 6. Compare weekly WR position evidence

Position is contextual rather than a permanent identity attribute. These checks compare weekly-stat WR labels with the same week's roster label and the player master's latest position.

In [9]:
wr_stats = player_stats.loc[
    (player_stats["position"] == "WR") & player_stats["player_id"].notna()
].copy()

roster_week_position = rosters_with_id[
    ["gsis_id", "season", "week", "team", "position", "status"]
].rename(columns={"position": "roster_position", "status": "roster_status"})

wr_position_comparison = wr_stats.merge(
    roster_week_position,
    left_on=["player_id", "season", "week", "team"],
    right_on=["gsis_id", "season", "week", "team"],
    how="left",
    validate="one_to_one",
).merge(
    players[["gsis_id", "position"]].rename(columns={"position": "master_position"}),
    left_on="player_id",
    right_on="gsis_id",
    how="left",
    validate="many_to_one",
    suffixes=("_roster_join", "_master_join"),
)

wr_roster_position_conflicts = wr_position_comparison.loc[
    wr_position_comparison["roster_position"].notna()
    & wr_position_comparison["roster_position"].ne("WR")
].copy()

position_conflict_summary = (
    wr_roster_position_conflicts.groupby(
        ["player_id", "player_display_name", "roster_position"],
        dropna=False,
    )
    .agg(
        conflicting_weeks=("week", "size"),
        first_season=("season", "min"),
        last_season=("season", "max"),
    )
    .sort_values("conflicting_weeks", ascending=False)
    .reset_index()
)

print(f"Weekly-stat rows labeled WR: {len(wr_stats):,}")
print(f"WR-stat rows with a non-WR roster label: {len(wr_roster_position_conflicts):,}")
display(position_conflict_summary)

master_position_counts = (
    wr_position_comparison["master_position"]
    .value_counts(dropna=False)
    .rename_axis("master_position")
    .reset_index(name="weekly_stat_rows_labeled_wr")
)
master_position_counts.head(10)

Weekly-stat rows labeled WR: 7,795
WR-stat rows with a non-WR roster label: 23


,player_id,player_display_name,roster_position,conflicting_weeks,first_season,last_season
0,00-0037091,Bo Melton,DB,13,2025,2025
1,00-0037745,Velus Jones Jr.,RB,7,2024,2025
2,00-0037450,Tyreik McAllister,RB,2,2024,2024
3,00-0038911,Malik Cunningham,QB,1,2023,2023


,master_position,weekly_stat_rows_labeled_wr
0,WR,7795


### 7. Compare participation evidence

A snap row is evidence that a player was included in game participation data. At least one positive snap is required for `participated_game`; an offensive snap greater than zero separately establishes offensive participation.

In [10]:
identified_wr_snaps = wr_mapped_snaps.loc[
    wr_mapped_snaps["identity_join"].eq("both")
].copy()
identified_wr_snaps["played_any_snap"] = (
    identified_wr_snaps[["offense_snaps", "defense_snaps", "st_snaps"]]
    .fillna(0)
    .gt(0)
    .any(axis=1)
)
identified_wr_snaps["played_offensive_snap"] = (
    identified_wr_snaps["offense_snaps"].fillna(0).gt(0)
)
identified_wr_snaps["played_other_snap_only"] = (
    ~identified_wr_snaps["played_offensive_snap"]
    & identified_wr_snaps[["defense_snaps", "st_snaps"]].fillna(0).gt(0).any(axis=1)
)

participation_summary = pd.DataFrame(
    {
        "check": [
            "identified WR snap rows",
            "at least one recorded snap",
            "at least one offensive snap",
            "defense or special teams only",
            "snap record with all snap counts zero",
        ],
        "rows": [
            len(identified_wr_snaps),
            identified_wr_snaps["played_any_snap"].sum(),
            identified_wr_snaps["played_offensive_snap"].sum(),
            identified_wr_snaps["played_other_snap_only"].sum(),
            (~identified_wr_snaps["played_any_snap"]).sum(),
        ],
    }
)
participation_summary["percent_of_identified_snap_rows"] = (
    participation_summary["rows"] / len(identified_wr_snaps)
).map(lambda value: f"{value:.2%}")

participation_summary

,check,rows,percent_of_identified_snap_rows
0,identified WR snap rows,8848,100.00%
1,at least one recorded snap,8847,99.99%
2,at least one offensive snap,8400,94.94%
3,defense or special teams only,447,5.05%
4,snap record with all snap counts zero,1,0.01%


In [11]:
weekly_stat_game_keys = player_stats_with_id[
    ["player_id", "game_id", "team"]
].drop_duplicates()

snap_to_stats = identified_wr_snaps.merge(
    weekly_stat_game_keys,
    left_on=["gsis_id", "game_id", "team"],
    right_on=["player_id", "game_id", "team"],
    how="left",
    validate="one_to_one",
    indicator="stats_join",
)

print(
    "Identified WR snap rows with a weekly-stat row: "
    f"{snap_to_stats['stats_join'].eq('both').mean():.2%}"
)
print(
    "Identified WR snap rows omitted by weekly stats: "
    f"{snap_to_stats['stats_join'].ne('both').sum():,}"
)

snap_to_stats.loc[
    snap_to_stats["stats_join"].ne("both"),
    ["season", "week", "game_id", "team", "player", "offense_snaps", "st_snaps"],
].head(20)

Identified WR snap rows with a weekly-stat row: 88.04%
Identified WR snap rows omitted by weekly stats: 1,058


,season,week,game_id,team,player,offense_snaps,st_snaps
5,2023,1,2023_01_ARI_WAS,WAS,Byron Pringle,0.0,1.0
15,2023,1,2023_01_BUF_NYJ,NYJ,Mecole Hardman,0.0,1.0
19,2023,1,2023_01_BUF_NYJ,BUF,Trent Sherfield,11.0,10.0
20,2023,1,2023_01_BUF_NYJ,BUF,Khalil Shakir,7.0,6.0
24,2023,1,2023_01_CAR_ATL,ATL,KhaDarel Hodge,7.0,13.0
33,2023,1,2023_01_CIN_CLE,CLE,Cedric Tillman,11.0,0.0
39,2023,1,2023_01_CIN_CLE,CIN,Trenton Irwin,12.0,0.0
40,2023,1,2023_01_CIN_CLE,CIN,Andrei Iosivas,1.0,25.0
76,2023,1,2023_01_HOU_BAL,BAL,Nelson Agholor,24.0,0.0
78,2023,1,2023_01_HOU_BAL,BAL,Tylan Wallace,0.0,16.0


In [12]:
wr_rosters = rosters_with_id.loc[rosters_with_id["position"] == "WR"].copy()
wr_roster_games = wr_rosters.merge(
    schedule_team_games,
    on=["season", "week", "team"],
    how="left",
    validate="many_to_one",
    indicator="schedule_join",
)

assert wr_roster_games["schedule_join"].eq("both").all()

identified_wr_snap_keys = identified_wr_snaps[
    ["gsis_id", "game_id", "team", "played_any_snap", "played_offensive_snap"]
].drop_duplicates(["gsis_id", "game_id", "team"])

wr_roster_participation = wr_roster_games.merge(
    identified_wr_snap_keys,
    on=["gsis_id", "game_id", "team"],
    how="left",
    validate="one_to_one",
    indicator="snap_join",
)

roster_status_participation = (
    wr_roster_participation.groupby("status", dropna=False)
    .agg(
        roster_rows=("gsis_id", "size"),
        snap_records=("snap_join", lambda values: values.eq("both").sum()),
        participated_games=("played_any_snap", lambda values: values.eq(True).sum()),
        offensive_participation=("played_offensive_snap", lambda values: values.eq(True).sum()),
    )
    .sort_values("roster_rows", ascending=False)
    .reset_index()
)

roster_status_participation.head(12)

,status,roster_rows,snap_records,participated_games,offensive_participation
0,ACT,8937,8844,8843,8399
1,DEV,4420,0,0,0
2,RES,1865,0,0,0
3,INA,1181,0,0,0
4,CUT,458,0,0,0
5,RET,182,0,0,0
6,EXE,7,0,0,0
7,TRC,5,0,0,0
8,TRD,2,0,0,0
9,PUP,1,0,0,0


### 8. Compare candidate WR player-week universes

The roster universe answers who was listed by a scheduled team, while the stats-plus-snaps union answers who generated football evidence. The latter is the recommended foundation for a football-fact table; roster status can be joined as context without generating inactive facts.

In [13]:
stats_wr_week_keys = wr_stats[
    ["player_id", "season", "week", "team", "game_id"]
].drop_duplicates()

snap_wr_week_keys = identified_wr_snaps[
    ["gsis_id", "season", "week", "team", "game_id"]
].rename(columns={"gsis_id": "player_id"}).drop_duplicates()

roster_wr_week_keys = wr_roster_games[
    ["gsis_id", "season", "week", "team", "game_id"]
].rename(columns={"gsis_id": "player_id"}).drop_duplicates()

active_roster_wr_week_keys = wr_roster_games.loc[
    wr_roster_games["status"] == "ACT",
    ["gsis_id", "season", "week", "team", "game_id"],
].rename(columns={"gsis_id": "player_id"}).drop_duplicates()

observed_wr_week_keys = pd.concat(
    [stats_wr_week_keys, snap_wr_week_keys],
    ignore_index=True,
).drop_duplicates()

for candidate in [
    stats_wr_week_keys,
    snap_wr_week_keys,
    roster_wr_week_keys,
    active_roster_wr_week_keys,
    observed_wr_week_keys,
]:
    assert not candidate.duplicated(WEEKLY_GRAIN).any()

candidate_universe_summary = pd.DataFrame(
    [
        ["weekly stats labeled WR", len(stats_wr_week_keys), stats_wr_week_keys["player_id"].nunique()],
        ["identified snap rows labeled WR", len(snap_wr_week_keys), snap_wr_week_keys["player_id"].nunique()],
        ["all weekly roster rows labeled WR", len(roster_wr_week_keys), roster_wr_week_keys["player_id"].nunique()],
        ["ACT roster rows labeled WR", len(active_roster_wr_week_keys), active_roster_wr_week_keys["player_id"].nunique()],
        ["recommended stats + identified snaps union", len(observed_wr_week_keys), observed_wr_week_keys["player_id"].nunique()],
    ],
    columns=["candidate_universe", "player_week_rows", "distinct_players"],
)

candidate_universe_summary

,candidate_universe,player_week_rows,distinct_players
0,weekly stats labeled WR,7795,340
1,identified snap rows labeled WR,8848,369
2,all weekly roster rows labeled WR,17058,646
3,ACT roster rows labeled WR,8937,372
4,recommended stats + identified snaps union,8881,371


### 9. Validate the proposed weekly grain

`game_id` is retained as an attribute. If a player/team/week points to more than one game, the proposed grain would be insufficient and the assertion below would fail.

In [14]:
observed_game_counts = (
    observed_wr_week_keys.groupby(WEEKLY_GRAIN, dropna=False)["game_id"]
    .nunique(dropna=False)
    .reset_index(name="game_count")
)

observed_same_week_team_counts = (
    observed_wr_week_keys.groupby(["player_id", "season", "week"], dropna=False)["team"]
    .nunique(dropna=False)
    .reset_index(name="team_count")
)

grain_validation = pd.DataFrame(
    {
        "check": [
            "duplicate observed rows at proposed grain",
            "proposed-grain keys linked to multiple games",
            "player-weeks linked to multiple teams",
            "null values in proposed grain",
        ],
        "failing_rows": [
            observed_wr_week_keys.duplicated(WEEKLY_GRAIN).sum(),
            observed_game_counts["game_count"].gt(1).sum(),
            observed_same_week_team_counts["team_count"].gt(1).sum(),
            observed_wr_week_keys[WEEKLY_GRAIN].isna().any(axis=1).sum(),
        ],
    }
)

assert grain_validation["failing_rows"].eq(0).all()

grain_validation

,check,failing_rows
0,duplicate observed rows at proposed grain,0
1,proposed-grain keys linked to multiple games,0
2,player-weeks linked to multiple teams,0
3,null values in proposed grain,0


### 10. Validate team changes between weeks

A player may represent multiple teams during a season. That is expected for trades and transactions; the critical rule is that the player is not assigned to multiple teams within the same week in the observed 2023–2025 data.

In [15]:
wr_player_season_teams = (
    observed_wr_week_keys.groupby(["player_id", "season"], dropna=False)
    .agg(
        team_count=("team", "nunique"),
        teams=("team", lambda values: ", ".join(sorted(set(values)))),
    )
    .reset_index()
)
wr_multi_team_seasons = wr_player_season_teams.loc[
    wr_player_season_teams["team_count"] > 1
].merge(
    players[["gsis_id", "display_name"]],
    left_on="player_id",
    right_on="gsis_id",
    how="left",
    validate="many_to_one",
)

print(f"Observed WR player-seasons with multiple teams: {len(wr_multi_team_seasons):,}")
print(
    "Observed player-weeks with multiple teams: "
    f"{observed_same_week_team_counts['team_count'].gt(1).sum():,}"
)

recognizable_trade_names = [
    "Amari Cooper",
    "Davante Adams",
    "DeAndre Hopkins",
    "Mecole Hardman",
    "Donovan Peoples-Jones",
]
recognizable_trade_ids = players.loc[
    players["display_name"].isin(recognizable_trade_names),
    "gsis_id",
]

trade_examples = (
    observed_wr_week_keys.loc[observed_wr_week_keys["player_id"].isin(recognizable_trade_ids)]
    .merge(
        players[["gsis_id", "display_name"]],
        left_on="player_id",
        right_on="gsis_id",
        how="left",
        validate="many_to_one",
    )
    [["player_id", "display_name", "season", "week", "team", "game_id"]]
    .sort_values(["display_name", "season", "week"])
)

display(wr_multi_team_seasons[["player_id", "display_name", "season", "team_count", "teams"]].head(20))
display(trade_examples.head(40))

Observed WR player-seasons with multiple teams: 39
Observed player-weeks with multiple teams: 0


,player_id,display_name,season,team_count,teams
0,00-0030035,Adam Thielen,2025,2,"MIN, PIT"
1,00-0030564,DeAndre Hopkins,2024,2,"KC, TEN"
2,00-0031236,Brandin Cooks,2025,2,"BUF, NO"
3,00-0031381,Davante Adams,2024,2,"LV, NYJ"
4,00-0031544,Amari Cooper,2024,2,"BUF, CLE"
5,00-0032211,Tyler Lockett,2025,2,"LV, TEN"
6,00-0032398,Chris Moore,2024,2,"ARI, WAS"
7,00-0033536,Mike Williams,2024,2,"NYJ, PIT"
8,00-0033943,Josh Reynolds,2024,2,"DEN, JAX"
9,00-0034272,Marquez Valdes-Scantling,2024,2,"BUF, NO"


,player_id,display_name,season,week,team,game_id
2,00-0031544,Amari Cooper,2023,1,CLE,2023_01_CIN_CLE
6,00-0031544,Amari Cooper,2023,2,CLE,2023_02_CLE_PIT
11,00-0031544,Amari Cooper,2023,3,CLE,2023_03_TEN_CLE
15,00-0031544,Amari Cooper,2023,4,CLE,2023_04_BAL_CLE
21,00-0031544,Amari Cooper,2023,6,CLE,2023_06_SF_CLE
25,00-0031544,Amari Cooper,2023,7,CLE,2023_07_CLE_IND
30,00-0031544,Amari Cooper,2023,8,CLE,2023_08_CLE_SEA
35,00-0031544,Amari Cooper,2023,9,CLE,2023_09_ARI_CLE
39,00-0031544,Amari Cooper,2023,10,CLE,2023_10_CLE_BAL
42,00-0031544,Amari Cooper,2023,11,CLE,2023_11_PIT_CLE


### 11. Record the validated Silver contracts

These are the decisions the next shared Silver prototype should implement. They are intentionally plain-language and testable.

In [16]:
silver_contracts = pd.DataFrame(
    [
        ["Canonical player ID", "Use exact GSIS identifiers and expose the field as player_id. Never join by player name."],
        ["Player metadata", "Use the player master first, retain populated source GSIS IDs that miss the master, and use source metadata only as an explicit fallback."],
        ["Snap identity bridge", "Map PFR to GSIS only through a one-to-one validated bridge. Preserve unresolved rows as exceptions."],
        ["Position", "Retain source-specific weekly positions; do not overwrite them with the player's latest master position."],
        ["WR validation membership", "Treat a week as WR-relevant when weekly stats or mapped snap counts label the player WR; use roster position as context."],
        ["Game participation", "participated_game is true only when total recorded offensive, defensive, or special-teams snaps are greater than zero."],
        ["Offensive participation", "offensive_participant is true only when offense_snaps is greater than zero."],
        ["Roster status", "Roster status is contextual evidence and does not by itself prove participation."],
        ["Weekly row universe", "Build football-fact rows from the union of identified weekly-stat and snap observations; enrich with roster data rather than generating all roster states as facts."],
        ["Weekly grain", "Require uniqueness at player_id + season + week + team. Retain game_id and opponent as attributes."],
        ["Postseason", "Retain detailed game_type and derive season_type = POST for WC, DIV, CON, and SB."],
        ["Missing values", "Keep missing evidence distinct from observed zero; do not fill absent snap or stat records with zero during joins."],
        ["Persistence", "Write no Silver output from this validation notebook; implement these contracts in the next shared Silver prototype."],
    ],
    columns=["contract", "validated_rule"],
)

silver_contracts

,contract,validated_rule
0,Canonical player ID,Use exact GSIS identifiers and expose the field as player_id. Never join by player name.
1,Player metadata,"Use the player master first, retain populated source GSIS IDs that miss the master, and use sour..."
2,Snap identity bridge,Map PFR to GSIS only through a one-to-one validated bridge. Preserve unresolved rows as exceptions.
3,Position,Retain source-specific weekly positions; do not overwrite them with the player's latest master p...
4,WR validation membership,Treat a week as WR-relevant when weekly stats or mapped snap counts label the player WR; use ros...
5,Game participation,"participated_game is true only when total recorded offensive, defensive, or special-teams snaps ..."
6,Offensive participation,offensive_participant is true only when offense_snaps is greater than zero.
7,Roster status,Roster status is contextual evidence and does not by itself prove participation.
8,Weekly row universe,Build football-fact rows from the union of identified weekly-stat and snap observations; enrich ...
9,Weekly grain,Require uniqueness at player_id + season + week + team. Retain game_id and opponent as attributes.


## Takeaways

- Exact GSIS identity is reliable for weekly stats and PBP receivers, while roster-only IDs missing from the current player master must remain visible as exceptions rather than being silently dropped.
- The player-master PFR bridge is one-to-one where populated, but it does not cover every snap identity. The WR exception is Cody White's `WhitCo05` snap identifier; no name-based repair is made here.
- Position is a weekly source attribute, not a permanent identity attribute. Observed disagreements demonstrate why the latest player-master position cannot define the historical WR slice by itself.
- Weekly stats are not participation-complete. Snap counts capture WRs who played without producing a weekly box-score row, and special-teams-only participation must remain distinct from offensive participation.
- Weekly roster snapshots are valuable enrichment for status and position, but including every roster row would turn reserve and development-squad listings into football-fact rows.
- The observed stats-plus-identified-snaps union passes the proposed `player_id + season + week + team` grain checks. Trades appear as team changes between weeks, with no same-week multi-team WR cases in the selected seasons.
- The next notebook can now prototype shared Silver player-week and game-participation tables using these rules, with automated assertions for identity coverage, join cardinality, and grain uniqueness.